# Modulo 7 · Lezione 2 — Preparazione dei dati per il ML

**Corso di Intelligenza Artificiale — Livello base**

---

## Cosa vediamo oggi?

| # | Argomento |
|---|------|
| 1 | La regola d'oro: fit su train, transform su tutto |
| 2 | Codifica delle variabili categoriche |
| 3 | Scalatura delle variabili numeriche |
| 4 | Imputazione dei valori mancanti |
| 5 | ColumnTransformer: elaborare colonne diverse insieme |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import (
    LabelEncoder, OrdinalEncoder, OneHotEncoder,
    StandardScaler, MinMaxScaler, RobustScaler
)
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

np.random.seed(42)
print("Librerie pronte")

---
## 1. La regola d'oro: fit solo su train

```
CONTAMINAZIONE DEI DATI (data leakage):
  Il modello apprende informazioni dal futuro (test) durante l'addestramento.
  → Il modello sembra migliore di quanto sia realmente in produzione.

SBAGLIATO (leakage):
  scaler.fit(X)              # usa TUTTO il dataset → contamina
  X_scaled = scaler.transform(X)
  X_tr, X_te = train_test_split(X_scaled)

CORRETTO:
  X_tr, X_te = train_test_split(X)  # dividere PRIMA
  scaler.fit(X_tr)                   # apprende le statistiche SOLO dal train
  X_tr_sc = scaler.transform(X_tr)   # applica al train
  X_te_sc = scaler.transform(X_te)   # applica al test (SENZA rifare il fit)

Stesso principio per: scaler, encoder, imputer, PCA...
```

In [ ]:
# Dimostrare l'effetto del leakage
np.random.seed(42)
X_demo = np.random.randn(100, 1) * 10 + 50  # media=50, std=10
# Aggiungiamo un outlier estremo solo nel test
X_demo[90] = 200

X_tr, X_te = train_test_split(X_demo, test_size=0.2, random_state=42)

# SBAGLIATO: fit su tutto → l'outlier del test contamina lo scaler
scaler_mal = StandardScaler()
scaler_mal.fit(X_demo)
print(f"Scaler SBAGLIATO (fit su tutto): media={scaler_mal.mean_[0]:.2f}, std={scaler_mal.scale_[0]:.2f}")

# CORRETTO: fit solo su train
scaler_bien = StandardScaler()
scaler_bien.fit(X_tr)
print(f"Scaler CORRETTO (fit su train): media={scaler_bien.mean_[0]:.2f}, std={scaler_bien.scale_[0]:.2f}")

print("\nLo scaler 'sbagliato' ha i suoi parametri influenzati dall'outlier del test.")
print("In produzione non avremo mai accesso a dati futuri → lo scaler CORRETTO è quello giusto.")

---
## 2. Codifica delle variabili categoriche

```
Gli algoritmi di ML lavorano con numeri → dobbiamo convertire il testo.

LABEL ENCODER:    A→0, B→1, C→2  (implica un ordine: C > B > A)
  ✓ Usarlo solo per la variabile TARGET (y)
  ✗ Non usarlo per le feature → introduce un ordine artificiale

ORDINAL ENCODER:  ['basso','medio','alto'] → [0, 1, 2]
  ✓ Usarlo quando esiste un ordine reale: taglia, livello di istruzione

ONE-HOT ENCODER:  'rosso' → [1,0,0], 'verde' → [0,1,0]
  ✓ Usarlo quando NON c'è ordine: città, colore, marca
  ⚠ Con molte categorie → molte colonne (valuta il target encoding)
```

In [ ]:
# Dataset di esempio
datos = pd.DataFrame({
    'nivel':    ['bajo', 'medio', 'alto', 'medio', 'bajo', 'alto'],
    'ciudad':   ['Madrid', 'Barcelona', 'Madrid', 'Valencia', 'Sevilla', 'Barcelona'],
    'compra':   ['si', 'no', 'si', 'si', 'no', 'si'],
})
print("Dati originali:")
print(datos)

# OrdinalEncoder per 'nivel' (ha un ordine reale)
ord_enc = OrdinalEncoder(categories=[['bajo', 'medio', 'alto']])
datos['nivel_cod'] = ord_enc.fit_transform(datos[['nivel']]).astype(int)

# OneHotEncoder per 'ciudad' (senza ordine)
ohe = OneHotEncoder(sparse_output=False, drop='first')  # drop='first' evita la multicollinearità
ciudad_ohe = ohe.fit_transform(datos[['ciudad']])
ciudad_df  = pd.DataFrame(ciudad_ohe, columns=ohe.get_feature_names_out(['ciudad']))

# LabelEncoder per il target 'compra'
le = LabelEncoder()
datos['compra_num'] = le.fit_transform(datos['compra'])
print(f"Classi LabelEncoder: {dict(zip(le.classes_, le.transform(le.classes_)))}")

resultado = pd.concat([datos, ciudad_df], axis=1)
print("\nDati codificati:")
print(resultado)

# Problema dell'ordine artificiale con LabelEncoder sulle feature
print("\n=== Perché NON usare LabelEncoder sulle feature? ===")
ciudades = ['Madrid', 'Barcelona', 'Valencia', 'Sevilla']
le_ciudad = LabelEncoder()
cod = le_ciudad.fit_transform(ciudades)
print(f"LabelEncoder: {dict(zip(ciudades, cod))}")
print("→ Sevilla=2, Valencia=3 → Valencia vale il 50% in più di Sevilla? Non ha senso!")

---
## 3. Scalatura delle variabili numeriche

```
Perché scalare?
  Senza scalatura: 'metri²' (50-200) domina su 'nº stanze' (1-5)
  → I gradienti in regressione logistica/SVM/KNN si distorcono

Quando non serve?
  Decision Tree e Random Forest → non sono influenzati dalla scalatura

STANDARD SCALER (Z-score):  x' = (x - μ) / σ  → media=0, std=1
  ✓ Uso generale, specialmente con distribuzione normale
  ✗ Sensibile agli outlier

MIN-MAX SCALER:  x' = (x - min) / (max - min)  → [0, 1]
  ✓ Quando serve un intervallo fisso di valori (reti neurali)
  ✗ Molto sensibile agli outlier

ROBUST SCALER:  x' = (x - mediana) / IQR  → robusto agli outlier
  ✓ Quando ci sono outlier che non puoi eliminare
```

In [ ]:
# Dataset con outlier
np.random.seed(42)
datos_num = np.array([10, 12, 11, 13, 10, 12, 100, 11, 13, 10], dtype=float)  # 100 è un outlier

scalers = {
    'StandardScaler': StandardScaler(),
    'MinMaxScaler':   MinMaxScaler(),
    'RobustScaler':   RobustScaler(),
}

print(f"Dati originali: {datos_num}")
print(f"{'Scaler':20} {'Valore 10':>12} {'Valore 12':>12} {'Outlier 100':>14}")
for nombre, sc in scalers.items():
    transformados = sc.fit_transform(datos_num.reshape(-1,1)).ravel()
    print(f"{nombre:20} {transformados[0]:>12.4f} {transformados[1]:>12.4f} {transformados[6]:>14.4f}")

# Visualizzazione
fig, axes = plt.subplots(1, 4, figsize=(16, 3))
titulos = ['Originale'] + list(scalers.keys())
datasets_viz = [datos_num] + [
    sc.fit_transform(datos_num.reshape(-1,1)).ravel()
    for sc in scalers.values()
]

for ax, titulo, d in zip(axes, titulos, datasets_viz):
    ax.boxplot(d, vert=True, patch_artist=True,
               boxprops=dict(facecolor='lightblue'),
               flierprops=dict(marker='o', color='red', ms=8))
    ax.set_title(titulo, fontsize=10)
    ax.set_xticklabels([])

plt.suptitle('Effetto della scalatura (punto rosso = outlier)', fontsize=11)
plt.tight_layout(); plt.show()

print("\nConclusione:")
print("  StandardScaler: l'outlier resta a 7σ — molto visibile ma altera gli altri valori")
print("  MinMaxScaler: tutti i valori normali si comprimono tra 0 e 0.03")
print("  RobustScaler: i valori normali mantengono la scala, l'outlier resta a ~9")

---
## 4. Imputazione dei valori mancanti

In [ ]:
# Dataset con valori mancanti
X_con_nan = np.array([
    [1.0, 2.0, np.nan],
    [3.0, np.nan, 4.0],
    [np.nan, 6.0, 7.0],
    [8.0, 8.0, 9.0],
    [5.0, 4.0, 3.0],
])

estrategias = ['mean', 'median', 'most_frequent', 'constant']
print("Dati originali:")
print(X_con_nan)
print()

for estrategia in estrategias:
    kwargs = {'fill_value': -999} if estrategia == 'constant' else {}
    imputer = SimpleImputer(strategy=estrategia, **kwargs)
    X_imp   = imputer.fit_transform(X_con_nan)
    print(f"Strategia '{estrategia}':")
    print(X_imp.round(2))
    print()

print("Regola di scelta della strategia:")
print("  mean:          dati numerici senza outlier")
print("  median:        dati numerici con outlier")
print("  most_frequent: dati categorici")
print("  constant:      quando il valore mancante ha un significato proprio (es: nessuna risposta)")

---
## 5. ColumnTransformer: tutto insieme

In [ ]:
# Dataset realistico: prevedere lo stipendio
np.random.seed(42)
n = 200
df_salario = pd.DataFrame({
    'experiencia': np.random.randint(0, 20, n).astype(float),
    'edad':        np.random.randint(22, 55, n).astype(float),
    'nivel_edu':   np.random.choice(['bachiller','grado','master','doctorado'], n),
    'ciudad':      np.random.choice(['Madrid','Barcelona','Valencia','Sevilla'], n),
    'salario':     None  # target
})
# Aggiungere valori mancanti
df_salario.loc[np.random.choice(n, 20, replace=False), 'experiencia'] = np.nan
df_salario.loc[np.random.choice(n, 10, replace=False), 'edad']        = np.nan

# Target: funzione del livello di istruzione e dell'esperienza
nivel_map = {'bachiller': 1, 'grado': 2, 'master': 3, 'doctorado': 4}
nivel_num = df_salario['nivel_edu'].map(nivel_map)
exp_clean = df_salario['experiencia'].fillna(df_salario['experiencia'].median())
df_salario['salario'] = (25000 + nivel_num * 8000 + exp_clean * 1500
                         + np.random.normal(0, 3000, n))

X = df_salario.drop('salario', axis=1)
y = df_salario['salario']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Definire cosa fare con ogni tipo di colonna
cols_numericas    = ['experiencia', 'edad']
cols_ordinales    = ['nivel_edu']
cols_nominales    = ['ciudad']

preprocesador = ColumnTransformer(transformers=[
    ('num',
     # Pipeline di passaggi per le numeriche
     # 1. Imputare i mancanti, 2. Scalare
     # (useremo Pipeline nella lezione 9; qui lo facciamo in due passaggi)
     StandardScaler(),
     cols_numericas),
    ('ord',
     OrdinalEncoder(categories=[['bachiller','grado','master','doctorado']]),
     cols_ordinales),
    ('nom',
     OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore'),
     cols_nominales),
], remainder='drop')  # 'passthrough' per mantenere il resto senza modifiche

# Fit solo su train, transform su entrambi
X_train_t = preprocesador.fit_transform(X_train)
X_test_t  = preprocesador.transform(X_test)

print(f"Forma originale X_train:      {X_train.shape}")
print(f"Forma trasformata X_train:  {X_train_t.shape}")
print(f"  Numeriche: {len(cols_numericas)} → {len(cols_numericas)} (scalate)")
print(f"  Ordinali: {len(cols_ordinales)} → {len(cols_ordinales)} (intero)")
print(f"  Nominali: {len(cols_nominales)} → {X_test_t.shape[1]-len(cols_numericas)-len(cols_ordinales)} (OHE con drop)")

print("\nPrime 3 righe trasformate:")
print(X_train_t[:3].round(3))

# Addestrare con i dati preprocessati
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
modelo = LinearRegression().fit(X_train_t, y_train)
r2 = r2_score(y_test, modelo.predict(X_test_t))
print(f"\nR² del modello con preprocessamento corretto: {r2:.4f}")

---
## Pratica

### Esercizio 1 — Codificare il dataset Titanic

**Consegna:** Carica il Titanic con seaborn e prepara le feature:

```python
import seaborn as sns
df = sns.load_dataset('titanic')[['survived','pclass','sex','age','fare','embarked','alone']].copy()
```

1. Imputa `age` con la mediana e `embarked` con la moda
2. Codifica `sex` con LabelEncoder e `embarked` con OneHotEncoder
3. Scala `age` e `fare` con StandardScaler
4. Dividi in train/test e verifica che lo scaler venga adattato solo sul train
5. Mostra la forma finale di X_train e X_test

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 1</summary>

```python
import seaborn as sns
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

df = sns.load_dataset('titanic')[['survived','pclass','sex','age','fare','embarked','alone']].copy()

# 1. Imputare
df['age']      = SimpleImputer(strategy='median').fit_transform(df[['age']]).ravel()
df['embarked'] = df['embarked'].fillna(df['embarked'].mode()[0])

# 2. Codificare sex
le = LabelEncoder()
df['sex_num'] = le.fit_transform(df['sex'])  # female=0, male=1

ohe = OneHotEncoder(sparse_output=False, drop='first')
emb_arr = ohe.fit_transform(df[['embarked']])
emb_df  = pd.DataFrame(emb_arr, columns=ohe.get_feature_names_out(['embarked']), index=df.index)

df_final = pd.concat([df[['survived','pclass','sex_num','age','fare','alone']], emb_df], axis=1)
df_final['alone'] = df_final['alone'].astype(int)

X = df_final.drop('survived', axis=1)
y = df_final['survived']

# 3 e 4. Split → fit dello scaler solo sul train
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

cols_escalar = ['age', 'fare']
scaler = StandardScaler()
X_tr[cols_escalar] = scaler.fit_transform(X_tr[cols_escalar])  # fit SUL TRAIN
X_te[cols_escalar] = scaler.transform(X_te[cols_escalar])      # solo transform sul test

# 5. Forma finale
print(f"X_train: {X_tr.shape}  |  X_test: {X_te.shape}")
print(f"Colonne: {list(X_tr.columns)}")
print(X_tr.head())
```
</details>

---
### Esercizio 2 — Confrontare gli scaler in un modello

**Consegna:** Sul dataset del diabete (`datasets.load_diabetes()`):

1. Addestra tre modelli con gli stessi dati ma con scalatura diversa: senza scalatura, StandardScaler, MinMaxScaler
2. Confronta l'R² sul test
3. Perché per LinearRegression la scalatura non cambia l'R²? E per quali modelli invece sarebbe importante?

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 2</summary>

```python
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import r2_score
import numpy as np

X, y = datasets.load_diabetes(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

resultados = {}

# Senza scalatura
m = LinearRegression().fit(X_tr, y_tr)
resultados['Senza scalatura'] = r2_score(y_te, m.predict(X_te))

# StandardScaler
sc = StandardScaler()
X_tr_s = sc.fit_transform(X_tr)
X_te_s = sc.transform(X_te)
m2 = LinearRegression().fit(X_tr_s, y_tr)
resultados['StandardScaler'] = r2_score(y_te, m2.predict(X_te_s))

# MinMaxScaler
mm = MinMaxScaler()
X_tr_m = mm.fit_transform(X_tr)
X_te_m = mm.transform(X_te)
m3 = LinearRegression().fit(X_tr_m, y_tr)
resultados['MinMaxScaler'] = r2_score(y_te, m3.predict(X_te_m))

for nombre, r2 in resultados.items():
    print(f"{nombre:20}: R² = {r2:.6f}")

print("\nSpiegazione:")
print("  LinearRegression: l'R² è UGUALE con o senza scalatura.")
print("  Motivo: scalare è una trasformazione lineare, gli iperpiani sono equivalenti.")
print("  I coefficienti cambiano, ma le predizioni finali sono identiche.")
print()
print("  Modelli in cui la scalatura È importante:")
print("  - KNN: usa le distanze, una feature grande domina")
print("  - SVM: il kernel RBF dipende dalle distanze")
print("  - LogisticRegression con regolarizzazione")
print("  - Reti neurali")
print("  - K-Means: usa le distanze euclidee")
```
</details>

---
### Esercizio 3 — ColumnTransformer completo

**Consegna:** Progetta un ColumnTransformer per questo dataset di prestiti:

```python
np.random.seed(10)
n = 300
df_prestamo = pd.DataFrame({
    'ingreso':    np.random.normal(35000, 12000, n).clip(10000),
    'edad':       np.random.randint(18, 70, n).astype(float),
    'deuda':      np.random.exponential(5000, n),
    'empleo':     np.random.choice(['empleado','autonomo','desempleado'], n),
    'educacion':  np.random.choice(['primaria','secundaria','universitaria'], n),
    'aprobado':   np.random.randint(0, 2, n),
})
# Aggiungere 15 valori mancanti in 'ingreso' e 'edad'
df_prestamo.loc[np.random.choice(n,15), 'ingreso'] = np.nan
df_prestamo.loc[np.random.choice(n,10), 'edad']    = np.nan
```

1. Numeriche (ingreso, edad, deuda): imputa con la mediana + StandardScaler
2. Nominale (empleo): OneHotEncoder
3. Ordinale (educacion): OrdinalEncoder con l'ordine corretto
4. Dividi in train/test e applica il ColumnTransformer
5. Addestra una LinearRegression e valutala

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 3</summary>

```python
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

np.random.seed(10)
n = 300
df = pd.DataFrame({
    'ingreso':   np.random.normal(35000, 12000, n).clip(10000),
    'edad':      np.random.randint(18, 70, n).astype(float),
    'deuda':     np.random.exponential(5000, n),
    'empleo':    np.random.choice(['empleado','autonomo','desempleado'], n),
    'educacion': np.random.choice(['primaria','secundaria','universitaria'], n),
    'aprobado':  np.random.randint(0, 2, n),
})
df.loc[np.random.choice(n,15), 'ingreso'] = np.nan
df.loc[np.random.choice(n,10), 'edad']    = np.nan

X = df.drop('aprobado', axis=1)
y = df['aprobado']
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 1. Numeriche: imputare + scalare
num_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

ct = ColumnTransformer([
    ('num', num_transformer, ['ingreso','edad','deuda']),
    ('nom', OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore'), ['empleo']),
    ('ord', OrdinalEncoder(categories=[['primaria','secundaria','universitaria']]), ['educacion']),
])

X_tr_t = ct.fit_transform(X_tr)
X_te_t = ct.transform(X_te)

print(f"X_train trasformata: {X_tr_t.shape}")
print(f"X_test trasformata:  {X_te_t.shape}")

modelo = LogisticRegression(max_iter=1000).fit(X_tr_t, y_tr)
print(f"\nAccuratezza sul test: {modelo.score(X_te_t, y_te):.4f}")
```
</details>

---
### Esercizio 4 — Individuare e correggere il leakage

**Consegna:** Il seguente codice contiene un errore di data leakage. Individualo, spiega perché è un problema e correggilo:

```python
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.datasets import load_diabetes

X, y = load_diabetes(return_X_y=True)

# Dove si trova il leakage?
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)  # ← PROBLEMA

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2)
modelo = LinearRegression().fit(X_train, y_train)
print(modelo.score(X_test, y_test))
```

In [ ]:
# ✏️ IL TUO CODICE QUI: spiega il problema e scrivi la versione corretta


<details>
<summary>💡 Vedi soluzione — Esercizio 4</summary>

```python
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

X, y = load_diabetes(return_X_y=True)

print("PROBLEMA nel codice originale:")
print("  scaler.fit_transform(X) usa TUTTI i dati (train + test).")
print("  → La media e la std dello scaler sono contaminate dai dati di test.")
print("  → Il modello 'vede' informazioni di test in anticipo.")
print("  In produzione i dati nuovi non esistono durante l'addestramento,")
print("  quindi le metriche sul test risultano troppo ottimistiche.")

# VERSIONE CON LEAKAGE (SBAGLIATA)
scaler_mal = StandardScaler()
X_scaled   = scaler_mal.fit_transform(X)
X_tr_mal, X_te_mal, y_tr, y_te = train_test_split(X_scaled, y, test_size=0.2, random_state=42)
r2_mal = LinearRegression().fit(X_tr_mal, y_tr).score(X_te_mal, y_te)

# VERSIONE CORRETTA
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
scaler_bien = StandardScaler()
X_tr_sc = scaler_bien.fit_transform(X_tr)  # fit SOLO sul train
X_te_sc = scaler_bien.transform(X_te)      # transform sul test

r2_bien = LinearRegression().fit(X_tr_sc, y_tr).score(X_te_sc, y_te)

print(f"\nR² CON leakage:    {r2_mal:.6f}")
print(f"R² SENZA leakage:    {r2_bien:.6f}")
print("\nIn questo caso la differenza è piccola (LinearRegression è stabile),")
print("ma con scaler molto sensibili agli outlier la differenza può essere significativa.")
```
</details>

---
### Esercizio 5 (sfida) — Feature engineering

**Consegna:** Il feature engineering consiste nel trasformare i dati affinché il modello apprenda meglio. Sul dataset delle case:

```python
np.random.seed(42)
n = 500
df_casas = pd.DataFrame({
    'metros':      np.random.uniform(40, 250, n),
    'habitaciones':np.random.randint(1, 7, n),
    'antiguedad':  np.random.randint(0, 50, n),
    'planta':      np.random.randint(0, 12, n),
    'zona':        np.random.choice(['A','B','C','D'], n),
})
df_casas['precio'] = (1800*df_casas['metros'] + 15000*df_casas['habitaciones']
                      - 500*df_casas['antiguedad'] + 2000*df_casas['planta']
                      + df_casas['zona'].map({'A':50000,'B':30000,'C':10000,'D':0})
                      + np.random.normal(0, 10000, n))
```

1. Crea feature derivate: `metros_por_hab = metros/habitaciones`, `es_nuevo = antiguedad < 5`, `zona_premium = zona in ['A','B']`
2. Preprocessa correttamente (ColumnTransformer)
3. Confronta l'R² con le feature originali rispetto a quelle arricchite
4. Di quanto migliora il modello grazie al feature engineering?

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 5</summary>

```python
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

np.random.seed(42)
n = 500
df = pd.DataFrame({
    'metros':       np.random.uniform(40, 250, n),
    'habitaciones': np.random.randint(1, 7, n),
    'antiguedad':   np.random.randint(0, 50, n),
    'planta':       np.random.randint(0, 12, n),
    'zona':         np.random.choice(['A','B','C','D'], n),
})
df['precio'] = (1800*df['metros'] + 15000*df['habitaciones']
                - 500*df['antiguedad'] + 2000*df['planta']
                + df['zona'].map({'A':50000,'B':30000,'C':10000,'D':0})
                + np.random.normal(0, 10000, n))

y = df['precio']

def preprocesar_y_evaluar(X, nombre):
    cols_num = X.select_dtypes(include='number').columns.tolist()
    cols_cat = X.select_dtypes(include='object').columns.tolist()
    ct = ColumnTransformer([
        ('num', StandardScaler(), cols_num),
        ('cat', OneHotEncoder(sparse_output=False, drop='first'), cols_cat),
    ]) if cols_cat else ColumnTransformer([('num', StandardScaler(), cols_num)])
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
    X_tr_t = ct.fit_transform(X_tr)
    X_te_t = ct.transform(X_te)
    modelo = LinearRegression().fit(X_tr_t, y_tr)
    r2 = r2_score(y_te, modelo.predict(X_te_t))
    print(f"{nombre:35}: R² = {r2:.4f}")
    return r2

# Versione originale
X_orig = df.drop('precio', axis=1)
r2_orig = preprocesar_y_evaluar(X_orig, 'Feature originali')

# Versione arricchita
df_eng = df.copy()
df_eng['metros_por_hab'] = df_eng['metros'] / df_eng['habitaciones']
df_eng['es_nuevo']       = (df_eng['antiguedad'] < 5).astype(int)
df_eng['zona_premium']   = df_eng['zona'].isin(['A','B']).astype(int)
X_eng = df_eng.drop('precio', axis=1)
r2_eng = preprocesar_y_evaluar(X_eng, 'Feature arricchite')

print(f"\nMiglioramento con il feature engineering: +{(r2_eng - r2_orig)*100:.2f} punti di R²")
```
</details>

---
## Riepilogo

```python
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

# REGOLA D'ORO: fit solo su train
X_tr, X_te = train_test_split(X)
scaler.fit(X_tr)              # apprende le statistiche dal train
X_tr_s = scaler.transform(X_tr)
X_te_s = scaler.transform(X_te)  # NESSUN nuovo fit

# ColumnTransformer: trasformazioni diverse per colonna
ct = ColumnTransformer([
    ('num', StandardScaler(),              ['col1', 'col2']),
    ('cat', OneHotEncoder(drop='first'),   ['col3']),
    ('ord', OrdinalEncoder(categories=[[...]]), ['col4']),
])
X_tr_t = ct.fit_transform(X_tr)
X_te_t = ct.transform(X_te)
```

## Cosa vediamo nella prossima lezione?
Nella **Lezione 3** applichiamo tutto ciò che abbiamo imparato per addestrare modelli di regressione e valutarli con metriche come MSE, MAE e R².